# Keras → PyTorch: LLM Training Primer

## Why this primer exists

Keras and PyTorch are both standard deep learning frameworks, but they sit at
different points on the convenience-vs-control spectrum.

**Keras** gives you `model.fit()` — one call that handles batching, gradient
computation, weight updates, and logging invisibly. This is ideal for standard
architectures (CNNs on images, tabular models) where minimal boilerplate matters.

**PyTorch** is the de facto toolkit for language model research and production.
Almost every published transformer, every open-source LLM (GPT, LLaMA, Mistral,
Gemma), and most production NLP systems are written in PyTorch. When you read a
model paper, clone a Hugging Face repo, or contribute to an LLM project, you are
reading PyTorch.

The insight this notebook builds: **for LLM training, Keras and PyTorch do
exactly the same things — the code just looks different.** Keras hides the training
loop inside `model.fit()`; PyTorch writes it out explicitly, giving you full
control over every step.

## What you will build

A **character-level language model** — the simplest version of what every LLM
does: given a window of text, predict the next character. Both the Keras and
PyTorch versions train on the same data with the same architecture and converge
to the same loss. The only differences are syntactic.

```
Input: "to be or not" (15 chars) → predict: " "
       "o be or not t" (15 chars) → predict: "o"
```

Architecture: `Embedding(vocab, 64) → LSTM(hidden=128) → Linear(128, vocab)`

## What you will be able to do when done

- Write a PyTorch training loop from scratch and understand each of the five lines
- Translate a Keras `Sequential` LM into a PyTorch `nn.Module`
- Know why `model.eval()` and `torch.no_grad()` are required before PyTorch inference
- Understand why PyTorch requires explicit `.to(device)` while Keras handles the GPU automatically
- Read PyTorch LLM training code from papers and GitHub without being surprised by the structure


## The Mental Model: Five Steps, Two Styles

Training a neural network is a five-step cycle repeated for every batch:

| Step | What happens |
|------|-------------|
| 1 | **Load a batch** — grab the next chunk of (context, next-char) pairs |
| 2 | **Forward pass** — run the batch through the model to get predictions |
| 3 | **Compute loss** — measure how wrong the predictions were (cross-entropy) |
| 4 | **Backward pass** — compute gradients via backpropagation |
| 5 | **Update weights** — apply the optimizer (Adam, SGD…) |

**Keras** runs all five steps inside `model.fit()`. You never see them written out.

**PyTorch** writes all five steps as explicit Python code — nothing is hidden:

```python
for X_batch, y_batch in loader:        # step 1: load a batch
    optimizer.zero_grad()               # prepare:  clear gradients from last batch
    logits = model(X_batch)             # step 2:   forward pass
    loss = criterion(logits, y_batch)   # step 3:   compute loss
    loss.backward()                     # step 4:   backpropagation
    optimizer.step()                    # step 5:   update weights
```

The same math happens in both cases. You will write these five lines in Section 6.


## Table of Contents

1. [Roadmap](#roadmap)
2. [Section 1: Setup](#section-1-setup)
3. [Section 2: The Dataset](#section-2-the-dataset)
4. [Section 3: Data Pipeline — Keras arrays vs PyTorch DataLoader](#section-3-data-pipeline)
5. [Section 4: Model Definition](#section-4-model-definition)
6. [Section 5: Compile vs Loss + Optimizer](#section-5-compile-vs-loss--optimizer)
7. [Section 6: Training — The Core Difference](#section-6-training)
8. [Section 7: Inference and Generation](#section-7-inference-and-generation)
9. [Cheat Sheet: Keras → PyTorch](#cheat-sheet)


## Roadmap

| Step | Keras | PyTorch | Core Difference |
|------|-------|---------|-----------------|
| Data | NumPy arrays → `fit()` | `Dataset` + `DataLoader` | PyTorch separates data loading from training |
| Model | `Sequential([...])` | `class M(nn.Module)` | Python class; shapes must be explicit |
| Loss + Opt | `model.compile(loss=..., optimizer=...)` | separate `criterion` + `optimizer` objects | No compile step |
| Training | `model.fit(X, y, epochs=...)` | explicit nested `for` loop | All five training steps are visible |
| Inference | `model.predict(x)` | `model.eval()` + `torch.no_grad()` + `model(x)` | Eval mode and grad suppression are manual |
| GPU | automatic | `.to(device)` on model and every batch | All device placement is explicit |


---
## Section 1: Setup

Both frameworks are installed in the `.venv` for this folder (`setup.ps1` / `setup.sh`).
The cell below confirms versions and imports everything used in this notebook.


In [ ]:
import subprocess, sys

required = ["torch", "numpy", "matplotlib", "tensorflow"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

print(f"PyTorch  {torch.__version__}")
print(f"TF/Keras {tf.__version__}")
print(f"NumPy    {np.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   {device}")


---
## Section 2: The Dataset

We take a short passage of text and build **(context, next_char)** training pairs.
A sliding window of `SEQ_LEN` characters is the input; the character that follows
is the label.

This is the same training objective used by every language model — GPT learns to
predict the next *token* from a context window; we predict the next *character*.
The math and the training loop are identical; only scale and architecture differ.

The data preparation below is **shared** — both Keras and PyTorch train on the
exact same sequences, which lets us compare their loss curves fairly.


In [ ]:
# ── shared data prep (used by both frameworks) ───────────────────────────
TEXT = (
    "to be or not to be that is the question "
    "whether tis nobler in the mind to suffer "
    "the slings and arrows of outrageous fortune "
    "or to take arms against a sea of troubles "
    "and by opposing end them to die to sleep "
    "no more and by a sleep to say we end "
)

chars     = sorted(set(TEXT))
char2idx  = {c: i for i, c in enumerate(chars)}
idx2char  = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(chars)
SEQ_LEN    = 15   # context window: 15 chars → predict the 16th

encoded   = np.array([char2idx[c] for c in TEXT], dtype=np.int32)
sequences = np.array([encoded[i : i + SEQ_LEN + 1] for i in range(len(encoded) - SEQ_LEN)])
X_np = sequences[:, :SEQ_LEN]   # (N, SEQ_LEN)  context
y_np = sequences[:, SEQ_LEN]    # (N,)           next character

print(f"Vocabulary ({VOCAB_SIZE} chars): {repr(''.join(chars))}")
print(f"Training sequences: {len(X_np)}")
print(f"X shape: {X_np.shape}   y shape: {y_np.shape}")
print()
ctx0 = "".join(idx2char[i] for i in X_np[0])
print(f"Example[0]: input = {repr(ctx0):22s}  target = {repr(idx2char[y_np[0]])}")
ctx4 = "".join(idx2char[i] for i in X_np[4])
print(f"Example[4]: input = {repr(ctx4):22s}  target = {repr(idx2char[y_np[4]])}")


---
## Section 3: Data Pipeline

This is the first visible API difference between Keras and PyTorch.

**Keras** is happy to receive plain NumPy arrays. Pass `X_np` and `y_np`
directly to `model.fit()` — batching is handled internally.

**PyTorch** separates data loading from training. You wrap your arrays in a
`Dataset` subclass (returns one sample at a time) and hand it to a `DataLoader`
(handles batching, shuffling, and optional multiprocess loading).

```
Keras  : model.fit(X_np, y_np, batch_size=32)     # arrays → fit() handles batching
PyTorch: Dataset → DataLoader → training loop     # explicit pipeline
```

The explicit pipeline is more code upfront, but it scales: the training loop
never changes regardless of whether your data comes from RAM, disk, or a web API.


In [ ]:
# PyTorch only — Keras accepts NumPy arrays directly in model.fit()

class CharDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(CharDataset(X_np, y_np), batch_size=32, shuffle=True)

# inspect one batch
X_b, y_b = next(iter(train_loader))
print(f"Batch X : {X_b.shape}   dtype={X_b.dtype}   (batch, SEQ_LEN)")
print(f"Batch y : {y_b.shape}   dtype={y_b.dtype}   (batch,)")
print(f"Batches per epoch: {len(train_loader)}")


---
## Section 4: Model Definition

Both models use the same architecture:

```
Embedding(VOCAB_SIZE, 64) → LSTM(hidden=128) → Linear(128, VOCAB_SIZE)
```

- **Embedding**: maps each integer character index to a 64-d dense vector (the "token embedding")
- **LSTM**: processes the 15-step sequence and produces a final hidden state summarising it
- **Linear**: maps the hidden state to one logit per possible next character

### Layer-by-layer correspondence

The two code cells below contain the identical architecture written in each style.
Here is the exact line-by-line mapping:

```python
# Keras                                               # PyTorch  (inside __init__)
keras.layers.Embedding(VOCAB_SIZE, 64, ...)      →   self.embed = nn.Embedding(vocab_size, 64)
keras.layers.LSTM(128)                           →   self.lstm  = nn.LSTM(64, 128, batch_first=True)
keras.layers.Dense(VOCAB_SIZE)                   →   self.head  = nn.Linear(128, vocab_size)
#                                                     # (wiring happens in forward(), not here)
```

What changed in each line:

| Layer | Keras | PyTorch | Why it changed |
|-------|-------|---------|----------------|
| Embedding | `Embedding(VOCAB_SIZE, 64)` | `nn.Embedding(vocab_size, 64)` | Same — no difference |
| LSTM | `LSTM(128)` — output size only | `nn.LSTM(64, 128, batch_first=True)` | Must give input size too; `batch_first=True` aligns dim order with Keras |
| Dense / Linear | `Dense(VOCAB_SIZE)` — output only | `nn.Linear(128, vocab_size)` | Must give input size too — PyTorch has no shape inference |
| LSTM output | last hidden state returned automatically | `_, (h_n, _) = self.lstm(x)` then `.squeeze(0)` | PyTorch returns `(output, (h_n, c_n))` — you choose what to keep |

### Keras — `Sequential`, shape inference

Define layers as a list; Keras chains them automatically. You only specify the *output* size of each layer.

### PyTorch — `nn.Module`, explicit shapes

Define a Python class. `__init__` declares layers as named attributes; `forward()` wires them as
ordinary Python code. You must specify **both** input and output sizes for every layer — there is
no shape inference.


In [ ]:
# ── Keras model ──────────────────────────────────────────────────────────
keras_model = keras.Sequential([
    keras.layers.Embedding(VOCAB_SIZE, 64, input_length=SEQ_LEN),
    keras.layers.LSTM(128),
    keras.layers.Dense(VOCAB_SIZE),   # raw logits; loss function handles softmax internally
], name="keras_lm")

keras_model.summary()


In [ ]:
# ── PyTorch model ────────────────────────────────────────────────────────
class TinyLM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # nn.LSTM: must give both input_size and hidden_size explicitly
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # nn.Linear: must give in_features explicitly — no shape inference
        self.head  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, SEQ_LEN)  — integer character ids
        x = self.embed(x)                   # → (batch, SEQ_LEN, embed_dim)
        _, (h_n, _) = self.lstm(x)          # h_n: (num_layers=1, batch, hidden_dim)
        logits = self.head(h_n.squeeze(0))  # → (batch, vocab_size)
        return logits


pt_model = TinyLM(VOCAB_SIZE).to(device)   # explicit device placement

n_params = sum(p.numel() for p in pt_model.parameters())
print(pt_model)
print(f"\nTotal parameters : {n_params:,}")
print(f"Running on       : {device}")


---
## Section 5: Compile vs Loss + Optimizer

In Keras, the loss function and optimizer are attached to the model at "compile" time.
In PyTorch there is no compile step — they are ordinary Python objects:

```python
# Keras — one call, everything attached to the model
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

# PyTorch — separate objects, no compile
criterion = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3)
```

`nn.CrossEntropyLoss()` and Keras `SparseCategoricalCrossentropy(from_logits=True)`
compute the same quantity: cross-entropy over integer class labels from raw logits.
The math is identical; the API differs.


In [ ]:
# ── Keras: compile ───────────────────────────────────────────────────────
keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
print("Keras model compiled.")


In [ ]:
# ── PyTorch: criterion + optimizer ───────────────────────────────────────
# No compile step. Both objects are independent — you pass them into the training loop yourself.
criterion = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(pt_model.parameters(), lr=1e-3)

print(f"criterion : {criterion}")
print(f"optimizer : {optimizer.__class__.__name__}")


---
## Section 6: Training — The Core Difference

`model.fit()` and the PyTorch training loop do **identical work**. The table below
is the exact mapping — every row is one thing `fit()` does internally, and the
PyTorch line that does the same thing explicitly.

### `model.fit()` — expanded line by line

| `model.fit()` does this internally… | PyTorch line that does the same |
|---|---|
| outer epoch loop (`epochs=40`) | `for epoch in range(40):` |
| sets model to training mode | `model.train()` |
| splits data into batches (`batch_size=32`) | `for X_b, y_b in train_loader:` |
| moves batch to the active device | `X_b, y_b = X_b.to(device), y_b.to(device)` |
| clears gradients from the previous batch | `optimizer.zero_grad()` |
| runs the forward pass | `logits = model(X_b)` |
| computes loss (using `loss=` arg from `compile()`) | `loss = criterion(logits, y_b)` |
| runs backpropagation | `loss.backward()` |
| updates weights (using `optimizer=` arg from `compile()`) | `optimizer.step()` |

In Keras, every row except the first runs invisibly inside `fit()`. In PyTorch you write each
one explicitly. That is the **entire** conceptual difference.

### Two lines that look new to Keras users

**`optimizer.zero_grad()`**
PyTorch accumulates gradients by default (useful for gradient checkpointing and some research
techniques). If you forget to clear them, gradients from previous batches add up and inflate
the updates, destabilising training.

**`model.train()`**
Layers like Dropout and BatchNorm behave differently during training vs inference. PyTorch requires
you to switch modes explicitly: `model.train()` before training, `model.eval()` before inference.
Keras switches automatically inside `fit()` and `predict()`.

### Keras

```python
history = model.fit(X, y, epochs=40, batch_size=32, validation_split=0.1)
```

One call. All nine rows above run internally for every batch of every epoch.

### PyTorch

You write all nine rows as explicit Python code — nothing hidden.


In [ ]:
# ── Keras: model.fit() ───────────────────────────────────────────────────
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # suppress TF info messages

keras_history = keras_model.fit(
    X_np, y_np,
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
# ── PyTorch: manual training loop ────────────────────────────────────────
pt_losses = []

for epoch in range(40):
    pt_model.train()        # training mode: activates Dropout / BatchNorm behaviour
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)     # explicit device placement for every batch
        y_batch = y_batch.to(device)

        optimizer.zero_grad()                    # clear old gradients
        logits = pt_model(X_batch)               # forward pass  → (batch, VOCAB_SIZE)
        loss   = criterion(logits, y_batch)      # cross-entropy loss
        loss.backward()                          # backpropagation
        optimizer.step()                         # update weights

        epoch_loss += loss.item() * len(X_batch)

    avg_loss = epoch_loss / len(X_np)
    pt_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d} | avg loss {avg_loss:.4f}")


In [ ]:
# ── Compare training curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(keras_history.history["loss"],     label="train", color="steelblue")
axes[0].plot(keras_history.history["val_loss"], label="val",   color="steelblue", linestyle="--")
axes[0].set_title("Keras — Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss"); axes[0].legend()

axes[1].plot(pt_losses, label="train", color="darkorange")
axes[1].set_title("PyTorch — Training Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Cross-entropy loss"); axes[1].legend()

plt.suptitle("Same architecture · same data · same optimiser → same loss curve shape")
plt.tight_layout()
plt.show()
print("Both models converge to similar loss. The difference is API style, not capability.")


---
## Section 7: Inference and Generation

Both models can generate text by repeatedly predicting the next character and
appending it to the running output (greedy decoding).

### The PyTorch inference pattern

Two things you must do manually that Keras handles automatically:

**`model.eval()`** — switches off layers that behave differently during training
(Dropout, BatchNorm). Forgetting this gives silently wrong predictions with no error.

**`torch.no_grad()`** — tells PyTorch not to build the computation graph for
backpropagation. Without it, every forward pass allocates memory for gradients you
will never use — wasting time and memory on long generation loops.

```python
# Keras — predict() handles both automatically
logits = model.predict(x)

# PyTorch — you call both yourself
model.eval()
with torch.no_grad():
    logits = model(x)
```


In [ ]:
def keras_generate(seed, length=60):
    # greedy next-character generation with the Keras model
    result = seed
    for _ in range(length):
        context = [char2idx.get(c, 0) for c in result[-SEQ_LEN:]]
        context = [0] * (SEQ_LEN - len(context)) + context      # left-pad if short
        x = np.array(context, dtype=np.int32)[np.newaxis, :]    # (1, SEQ_LEN)
        logits = keras_model.predict(x, verbose=0)              # (1, VOCAB_SIZE)
        next_char = idx2char[int(np.argmax(logits[0]))]
        result += next_char
    return result


In [ ]:
def pytorch_generate(seed, length=60):
    # greedy next-character generation with the PyTorch model
    pt_model.eval()                      # switch to inference mode
    result = seed
    with torch.no_grad():                # no gradient tracking — saves memory
        for _ in range(length):
            context = [char2idx.get(c, 0) for c in result[-SEQ_LEN:]]
            context = [0] * (SEQ_LEN - len(context)) + context
            x = torch.tensor(context, dtype=torch.long).unsqueeze(0).to(device)
            logits = pt_model(x)                            # (1, VOCAB_SIZE)
            next_char = idx2char[int(logits.argmax(dim=-1).item())]
            result += next_char
    return result


In [ ]:
# ── Generate from both models with the same seed ────────────────────────
SEED = "to be or not"

keras_out   = keras_generate(SEED, length=60)
pytorch_out = pytorch_generate(SEED, length=60)

print(f"Seed    : {repr(SEED)}")
print()
print(f"Keras  : {repr(keras_out)}")
print()
print(f"PyTorch: {repr(pytorch_out)}")


---
## Cheat Sheet: Keras → PyTorch

### API mapping

| Task | Keras | PyTorch |
|------|-------|---------|
| Import | `import keras` | `import torch; import torch.nn as nn` |
| Model | `keras.Sequential([...])` | `class M(nn.Module): __init__ + forward` |
| Embedding | `keras.layers.Embedding(vocab, dim)` | `nn.Embedding(vocab, dim)` |
| LSTM | `keras.layers.LSTM(units)` | `nn.LSTM(in_size, hidden, batch_first=True)` |
| Dense / Linear | `keras.layers.Dense(out)` | `nn.Linear(in_features, out_features)` |
| Loss | `compile(loss="sparse_categorical_crossentropy")` | `criterion = nn.CrossEntropyLoss()` |
| Optimizer | `compile(optimizer="adam")` | `optimizer = torch.optim.Adam(model.parameters())` |
| Train | `model.fit(X, y, epochs=...)` | `for epoch: for batch in loader: zero_grad / forward / loss / backward / step` |
| Predict | `model.predict(x)` | `model.eval()` + `torch.no_grad()` + `model(x)` |
| GPU | automatic | `model.to(device)` once; `batch.to(device)` every iteration |
| Save | `model.save_weights(path)` | `torch.save(model.state_dict(), path)` |
| Load | `model.load_weights(path)` | `model.load_state_dict(torch.load(path))` |

### Gotchas

| | What goes wrong | Fix |
|--|----------------|-----|
| Forgot `optimizer.zero_grad()` | Gradients accumulate → exploding updates | Call at the start of every batch |
| Forgot `model.eval()` | Dropout stays on during inference → stochastic outputs | Call before any inference loop |
| Forgot `torch.no_grad()` | Memory grows during inference; slower | Wrap inference in `with torch.no_grad():` |
| Forgot `.to(device)` on a batch | RuntimeError: tensors on different devices | Move model once on init; move every batch in the loop |
| LSTM output unpacking | `lstm()` returns `(output, (h_n, c_n))` — must unpack | Use `_, (h_n, _) = self.lstm(x)` to grab only the final hidden state |

### The five lines at the heart of every PyTorch training loop

```python
optimizer.zero_grad()              # clear gradients from last batch
logits = model(X_batch)            # forward pass
loss = criterion(logits, y_batch)  # compute loss
loss.backward()                    # backpropagation
optimizer.step()                   # update weights
```

### Where to go next

- `learning/genai/01-rnns/` — LSTM in depth, sequence-to-sequence models
- `learning/genai/02-transformers/` — build a full Transformer in PyTorch
- Hugging Face `transformers` — fine-tune real LLMs with exactly this loop structure


---
## What Changed, What Didn't

**Keras and PyTorch are the same for LLM training:**

- The math is identical — same gradient descent, same loss, same parameter updates
- Architecture primitives map 1-to-1: `Embedding`, `LSTM`, `Linear` exist in both
- The training objective (next-token prediction) is identical to GPT
- Adam, learning rates, batch sizes — all hyperparameters carry over directly

**What is different:**

| | Keras | PyTorch |
|--|-------|---------|
| Training loop | hidden in `fit()` | explicit Python code — every step visible |
| Device management | automatic | explicit `.to(device)` everywhere |
| Inference mode | automatic in `predict()` | manual `eval()` + `no_grad()` |
| Data loading | pass NumPy arrays | `Dataset` + `DataLoader` |
| Model definition | configuration list | Python class with `forward()` |

**Why PyTorch dominates LLM research:**

When you are experimenting with new architectures, custom loss functions, or novel
training procedures (RLHF, DPO, gradient accumulation, speculative decoding), having
the full training loop as readable Python code means you can modify exactly one thing —
and know exactly what changed. Keras is the right choice when you want minimal code
for standard problems; PyTorch is the right choice when you need control.

For LLM training specifically, the difference in *code written* is small. The difference
in *control available* is large.
